# RG3 gas: continuous normal-rank fusion and internal-only threshold selection
Reuses 42 raw outer/inner predictions, no refit. Monotone interpolation plus exponential tail continuation prevents CDF steps/saturation discarding score separation.
Six prespecified internal FPR rates, eight fusion functions. Pooled-F1 and macro-positive-block-F1 selectors restricted to <=1% internal rate; no outer-label selection.
Raw dataset and original 200-tree baseline unchanged; original baseline decisions at internal1% must reproduce exactly.
Fixed outer-best candidate is exploratory, not independently selected/validated. No metric guarantee.

In [ ]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
outer=pd.read_csv(tables/'rg3_rank_fusion_predictions.csv',parse_dates=['TimeStamp'])
outer=outer.loc[outer.model.eq('baseline_rank')].copy()
inside=pd.read_csv(tables/'rg3_rank_fusion_inner_scores.csv',parse_dates=['TimeStamp'])
def transform(scores,normal):
    out=[]
    for k in range(2):
        ref=np.sort(normal[:,k]);x=np.unique(ref)
        levels=(np.searchsorted(ref,x,side='right')+.5)/(len(ref)+1.)
        v=np.interp(scores[:,k],x,levels)
        scale=max(float(np.quantile(ref,.75)-np.quantile(ref,.25)),.01)
        low=scores[:,k]<x[0];high=scores[:,k]>x[-1]
        v[low]=levels[0]*np.exp(np.maximum((scores[low,k]-x[0])/scale,-700))
        v[high]=1-(1-levels[-1])*np.exp(np.maximum(-(scores[high,k]-x[-1])/scale,-700))
        out.append(v)
    return np.column_stack(out)
def fusion(u):
    a=u[:,0];b=u[:,1]
    return {'baseline_rank':a,'expert_rank':b,'mean25':.75*a+.25*b,'mean50':.5*a+.5*b,'mean75':.25*a+.75*b,'geometric':np.sqrt(a*b),'rank_min':np.minimum(a,b),'rank_max':np.maximum(a,b)}
def stats(yy,pr):
    tn,fp,fn,tp=confusion_matrix(yy,pr,labels=[0,1]).ravel()
    return int(tp),int(fp),int(fn),int(tn),2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.
records=[];choices=[];candidate_logs=[]
rates=[.0025,.005,.0075,.01,.015,.02]
for (scheme,b),q in outer.groupby(['scheme','block'],sort=False):
    train=inside.loc[inside.scheme.eq(scheme)&inside.heldout_block.eq(b)].copy()
    raw=train[['raw_baseline_score','raw_expert_score']].to_numpy(float)
    valid=np.isfinite(raw).all(axis=1);train=train.loc[valid].reset_index(drop=True);raw=raw[valid]
    normal=raw[train.actual.eq(0)]
    cal=fusion(transform(raw,normal))
    scores=fusion(transform(q[['raw_baseline_score','raw_expert_score']].to_numpy(float),normal))
    candidates=[]
    for name,v in cal.items():
        for rate in rates:
            cut=float(np.quantile(v[train.actual.eq(0)],1-rate,method='higher'))
            pr=v>cut
            tp,fp,fn,tn,f1=stats(train.actual,pr)
            block_f1=[]
            for ib,ix in train.groupby('block').groups.items():
                ids=np.array(list(ix))
                if train.actual.iloc[ids].sum():block_f1.append(stats(train.actual.iloc[ids],pr[ids])[-1])
            macro=float(np.mean(block_f1)) if block_f1 else 0.
            candidate=dict(model=name,budget=rate,threshold=cut,inner_F1=f1,inner_macro_F1=macro,inner_TP=tp,inner_FP=fp)
            candidates.append(candidate);candidate_logs.append(dict(scheme=scheme,heldout_block=b,**candidate))
            for j,r in enumerate(q.itertuples()):records.append(dict(scheme=scheme,model=name,budget=rate,TimeStamp=r.TimeStamp,block=b,actual=int(r.actual),score=float(scores[name][j]),threshold=cut,prediction=int(scores[name][j]>cut)))
    # Restrict deployment selectors to <=1% INTERNAL normal rates.
    eligible=[c for c in candidates if c['budget']<=.01]
    for label,key in [('inner_F1_selected','inner_F1'),('inner_macro_selected','inner_macro_F1')]:
        selected=max(eligible,key=lambda c:(c[key],-c['inner_FP'],c['model']=='baseline_rank',-abs(c['budget']-.01)))
        if selected[key]==0:selected=next(c for c in eligible if c['model']=='baseline_rank' and c['budget']==.01)
        choices.append(dict(scheme=scheme,block=b,selector=label,**selected))
        for j,r in enumerate(q.itertuples()):records.append(dict(scheme=scheme,model=label,budget=-1.,TimeStamp=r.TimeStamp,block=b,actual=int(r.actual),score=float(scores[selected['model']][j]),threshold=selected['threshold'],prediction=int(scores[selected['model']][j]>selected['threshold'])))
pp=pd.DataFrame(records);rows=[]
for (scheme,name,rate),g in pp.groupby(['scheme','model','budget'],sort=False):
    tp,fp,fn,tn,f1=stats(g.actual,g.prediction)
    rows.append(dict(scheme=scheme,model=name,budget=rate,TP=tp,FP=fp,FN=fn,TN=tn,F1=f1,AP=average_precision_score(g.actual,g.score)))
metrics=pd.DataFrame(rows)
pp.to_csv(tables/'rg3_smooth_rank_fusion_predictions.csv',index=False,encoding='utf-8-sig')
metrics.to_csv(tables/'rg3_smooth_rank_fusion_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(choices).to_csv(tables/'rg3_smooth_rank_fusion_inner_choices.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(candidate_logs).to_csv(tables/'rg3_smooth_rank_fusion_inner_candidates.csv',index=False,encoding='utf-8-sig')
old=pd.read_csv(tables/'rg3_nested_ensemble_predictions.csv',parse_dates=['TimeStamp'])
check=pp.loc[pp.model.eq('baseline_rank')&pp.budget.eq(.01)].merge(old.loc[old.model.eq('max_union'),['scheme','TimeStamp','predicted_train_fpr1']],on=['scheme','TimeStamp'],validate='one_to_one')
assert len(check)==1023 and (check.prediction.astype(bool)==check.predicted_train_fpr1).all()
print('Original baseline decisions preserved:',len(check))
for scheme in metrics.scheme.unique():
    m=metrics.loc[metrics.scheme.eq(scheme)]
    print(scheme,m.sort_values(['F1','FP'],ascending=[False,True]).head(10).to_string(index=False))
    print('TRAIN-ONLY SELECTORS',m.loc[m.model.str.startswith('inner_')].to_string(index=False))
    print('BALANCED TARGET CANDIDATES',m.loc[(m.TP>=4)&(m.FP<=5)|(m.TP>=5)&(m.FP<=6)].to_string(index=False))


## Executed results
Original baseline1023 decisions verified. Best fixed whole rank_max TP5 FP11 F1=0.303. Both internal selectors whole TP2 FP7 F1=0.154; forward TP0 FP5 F1=0. No balanced target candidate.
See reports/rg3_iterative_improvement_log.md for protocol, limitations and source tables.